<a href="https://colab.research.google.com/github/Akhilesh-maker-creator/SnakeClassifier/blob/main/SnakeClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision

In [2]:
!pip install kaggle

In [3]:
from google.colab import files
files.upload()  # upload kaggle.json here

Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"akhileshsinghrawat","key":"23e82829555ee2f1b34d3a317175d59b"}'}

In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d sameeharahman/preprocessed-snake-images

Dataset URL: https://www.kaggle.com/datasets/sameeharahman/preprocessed-snake-images
License(s): DbCL-1.0
preprocessed-snake-images.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
!unzip preprocessed-snake-images.zip -d dataset

Archive:  preprocessed-snake-images.zip
replace dataset/preprocessed-cleaned-set/train/class-1/00056e9548477cda7a885bb423cb668c.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [19]:
import os
import shutil
import random

random.seed(42)

base_dir = "dataset/preprocessed-cleaned-set"
train_dir = os.path.join(base_dir, "train")
val_dir   = os.path.join(base_dir, "val")
test_dir  = os.path.join(base_dir, "test")

os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name)

    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)
    random.shuffle(images)

    total = len(images)
    val_split = int(0.15 * total)
    test_split = int(0.15 * total)

    os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)

    # Move to VAL
    for img in images[:val_split]:
        shutil.move(
            os.path.join(class_path, img),
            os.path.join(val_dir, class_name, img)
        )

    # Move to TEST
    for img in images[val_split:val_split + test_split]:
        shutil.move(
            os.path.join(class_path, img),
            os.path.join(test_dir, class_name, img)
        )

print("✅ Dataset split complete")

✅ Dataset split complete


In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:

!cp -r dataset /content/drive/MyDrive/dataset

y
ah;lkdlfl
^C


In [27]:
import torch
from torchvision import datasets, transforms, models
from torch import nn, optim
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from PIL import Image

In [28]:
torch.manual_seed(42)
np.random.seed(42)

In [29]:
train_dir = "dataset/preprocessed-cleaned-set/train"
val_dir   = "dataset/preprocessed-cleaned-set/val"
test_dir  = "dataset/preprocessed-cleaned-set/test"

In [30]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [31]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir, transform=val_transform)
test_dataset  = datasets.ImageFolder(test_dir, transform=val_transform)

class_names = train_dataset.classes

In [32]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.efficientnet_b0(pretrained=True)

for param in model.features.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    len(class_names)
)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 192MB/s]


In [35]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [36]:
epochs = 15
best_val_acc = 0
patience = 3
counter = 0

for epoch in range(epochs):
    model.train()
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()

    train_acc = correct / len(train_dataset)

    # VALIDATION
    model.eval()
    val_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            val_correct += (preds == labels).sum().item()

    val_acc = val_correct / len(val_dataset)

    print(f"Epoch {epoch+1}: Train {train_acc:.4f}, Val {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered")
        break

Epoch 1: Train 0.6147, Val 0.7016
Epoch 2: Train 0.6805, Val 0.7138
Epoch 3: Train 0.6887, Val 0.7269
Epoch 4: Train 0.6996, Val 0.7311
Epoch 5: Train 0.7017, Val 0.7326
Epoch 6: Train 0.7001, Val 0.7392
Epoch 7: Train 0.6982, Val 0.7315
Epoch 8: Train 0.6986, Val 0.7403
Epoch 9: Train 0.6949, Val 0.7365
Epoch 10: Train 0.7016, Val 0.7434
Epoch 11: Train 0.6985, Val 0.7480
Epoch 12: Train 0.7028, Val 0.7380
Epoch 13: Train 0.7047, Val 0.7242
Epoch 14: Train 0.7032, Val 0.7369
Early stopping triggered


In [37]:
model.load_state_dict(torch.load("best_model.pth"))

for param in model.features[-3:].parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(3):
    model.train()
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()

    print(f"Fine-tune Epoch {epoch+1}: {correct / len(train_dataset):.4f}")

Fine-tune Epoch 1: 0.7531
Fine-tune Epoch 2: 0.8200
Fine-tune Epoch 3: 0.8508


In [38]:
model.eval()
val_correct = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        val_correct += (preds == labels).sum().item()

val_acc = val_correct / len(val_dataset)

print(f"Epoch {epoch+1}: Train {train_acc:.4f}, Val {val_acc:.4f}")

Epoch 3: Train 0.7032, Val 0.8562


In [39]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

test_acc = np.mean(np.array(all_preds) == np.array(all_labels))
print("Test Accuracy:", test_acc)

Test Accuracy: 0.8668968162639049


In [40]:
print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=class_names))

[[383   7  14  46  13]
 [ 12 465  43  28   1]
 [ 11  23 490   5   3]
 [ 50  30  14 402  21]
 [ 11   1   5   9 520]]
              precision    recall  f1-score   support

     class-1       0.82      0.83      0.82       463
     class-2       0.88      0.85      0.87       549
     class-3       0.87      0.92      0.89       532
     class-4       0.82      0.78      0.80       517
     class-5       0.93      0.95      0.94       546

    accuracy                           0.87      2607
   macro avg       0.86      0.87      0.86      2607
weighted avg       0.87      0.87      0.87      2607



In [42]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, "snake_model.pth")

In [43]:
from google.colab import files

uploaded = files.upload()

Saving western-diamondback-rattlesnake-crotalus-atrox-600nw-2571283237.webp to western-diamondback-rattlesnake-crotalus-atrox-600nw-2571283237.webp


In [44]:
def predict_image(image_path):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)

    return class_names[pred.item()], conf.item()

for img in uploaded.keys():
    pred, conf = predict_image(img)
    print(f"{img} → {pred} ({conf:.2f})")

western-diamondback-rattlesnake-crotalus-atrox-600nw-2571283237.webp → class-5 (1.00)
